# 04 — Perfiles de contratación y valores atípicos

Comparamos volumen y composición de categorías en el corte descriptivo 2023–2024. El ruido y la distancia al centroide describen perfiles en este conjunto; **no prueban irregularidades**. Este cuaderno lee resultados publicados y no los modifica.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent
result = ROOT / "artifacts" / "clustering"
if not (result / "_SUCCESS").is_file():
    raise FileNotFoundError("Construya los resultados según docs/ComandosEjecutarOrdenReplica.md")
meta = json.loads((result / "_metadata.json").read_text(encoding="utf-8"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.features.build_part1 import directory_fingerprint  # noqa: E402

part1 = ROOT / "data" / "processed" / "part1"
if (
    not (part1 / "_SUCCESS").is_file()
    or directory_fingerprint(part1) != meta["source"]["part1_sha256"]
):
    raise ValueError("Resultados desactualizados: vuelva a ejecutar el clustering.")
features = pd.read_parquet(result / "features.parquet")
assignments = pd.read_parquet(result / "assignments.parquet")
km = pd.read_parquet(result / "kmeans_metrics.parquet")
hd = pd.read_parquet(result / "hdbscan_metrics.parquet")
profiles = pd.read_parquet(result / "profiles.parquet")
print(
    "Ventana:",
    meta["source_window"],
    "| Entidades:",
    meta["input_rows"],
    "| Adjudicaciones:",
    meta["source_awards"],
)

## 1. Referencia y datos

Los cuantiles resumen el universo observado. Las proporciones corresponden a categorías amplias; tres proporciones suman uno y no son variables independientes. Los valores se escalaron en la ventana completa para este estudio descriptivo.


In [ ]:
display(features[["award_count", "fraction_0", "fraction_1", "fraction_2"]].describe())
print(
    "Vectores originales distintos:",
    len(features[["log_award_count", "fraction_0", "fraction_1", "fraction_2"]].drop_duplicates()),
)
features["award_count"].plot.hist(
    bins=50, logy=True, title="Adjudicaciones por entidad: escala logarítmica en frecuencia"
)
plt.xlabel("Número de adjudicaciones")
plt.show()

## 2. KMeans++

La opción provisional maximiza la silueta de una muestra de hasta 1 500 entidades en la variante ponderada, descartando grupos menores al 1 % o a 15 entidades. Revisar también tamaños, estabilidad y escala: la regla numérica no demuestra que exista una única tipología correcta.


In [ ]:
display(km.sort_values(["variant", "k"]))
print("Selección provisional:", meta["kmeans_selected"])
display(profiles[profiles.method == "kmeans"].sort_values("label"))
display(pd.DataFrame(meta["stability"]))
display(pd.DataFrame(meta["scaling_comparison"]))
for name, group in km.groupby("variant"):
    plt.plot(group.k, group.silhouette, marker="o", label=name)
plt.xlabel("k")
plt.ylabel("Silueta de muestra")
plt.legend()
plt.show()

## 3. HDBSCAN y anomalías

La silueta de HDBSCAN excluye ruido y debe leerse junto con cobertura. La selección provisional requiere al menos dos grupos y cobertura de 50 %. Revisamos también los vectores idénticos dentro de cada grupo: muchas entidades con características iguales pueden formar grupos densos sin que ello indique un mercado real.


In [ ]:
display(hd.sort_values(["variant", "min_cluster_size", "min_samples"]))
print("Selección provisional:", meta["hdbscan_selected"])
selected = assignments[assignments.selected].merge(features, on="entity_id", validate="many_to_one")
density = selected[(selected.method == "hdbscan") & (selected.label >= 0)]
dims = ["log_award_count", "fraction_0", "fraction_1", "fraction_2"]
if not density.empty:
    distinct = density.groupby("label", group_keys=False)[dims].apply(
        lambda x: len(x.drop_duplicates())
    )
    print("Grupos con un único vector distinto:", int((distinct == 1).sum()), "/", len(distinct))
    print("Entidades en esos grupos:", int(density.label.isin(distinct[distinct == 1].index).sum()))
display(
    selected.groupby("method").agg(
        entidades=("entity_id", "size"), candidatos=("outlier_candidate", "sum")
    )
)
display(
    selected[selected.outlier_candidate][
        ["entity_id", "method", "label", "award_count", "fraction_0", "fraction_1", "fraction_2"]
    ].head(30)
)

## 4. Referencia por volumen y coincidencia de señales

Comparamos los grupos con cuatro tramos definidos por los cuartiles del número de adjudicaciones. Los empates permanecen juntos y pueden reducir el número de tramos; esto no es otro modelo. La intersección de las señales de KMeans y HDBSCAN se muestra en el mismo universo, sin convertirla en una tasa de irregularidad.

In [ ]:
import numpy as np

baseline = features.copy()
baseline["volume_band"] = pd.qcut(baseline.award_count, q=4, duplicates="drop").astype(str)
k_selected = selected[selected.method == "kmeans"].copy()
h_selected = selected[selected.method == "hdbscan"].copy()
if not k_selected.empty:
    volume_compare = k_selected.merge(baseline[["entity_id", "volume_band"]], on="entity_id")
    display(pd.crosstab(volume_compare.label, volume_compare.volume_band))
    display(
        volume_compare.groupby("label")[
            ["award_count", "fraction_0", "fraction_1", "fraction_2"]
        ].quantile([0.25, 0.5, 0.75])
    )
if not k_selected.empty and not h_selected.empty:
    flags = k_selected[["entity_id", "outlier_candidate"]].merge(
        h_selected[["entity_id", "outlier_candidate"]], on="entity_id", suffixes=("_k", "_h")
    )
    display(pd.crosstab(flags.outlier_candidate_k, flags.outlier_candidate_h))
    examples = k_selected[k_selected.outlier_candidate].sort_values("distance", ascending=False)
    display(
        examples[
            [
                "entity_id",
                "label",
                "distance",
                "award_count",
                "fraction_0",
                "fraction_1",
                "fraction_2",
            ]
        ].head(5)
    )

## 5. Jerárquico: Ward sobre una muestra estratificada

Seleccionamos hasta 400 entidades con semilla 42 y cupos equilibrados entre tramos de volumen construidos con los cuartiles. Los empates permanecen juntos; si un tramo tiene menos de 100 entidades se redistribuyen los cupos entre los demás. Guardamos los ID en la tabla de la celda, no como muestra poblacional representativa. Ward usa distancia euclidiana en las cuatro variables con el mismo escalado ponderado de KMeans. El dendrograma resume fusiones de esta muestra: sus alturas no son montos ni probabilidades.

Para comparar con KMeans cortamos el árbol en cuatro grupos, por la selección previa de KMeans, no porque Ward demuestre que cuatro sea el número verdadero. La comparación usa solo los mismos ID y ARI, que no depende del número asignado a cada etiqueta.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.metrics import adjusted_rand_score

# Asignación equilibrada, con redistribución determinista si un tramo se agota.
parts = [g.sort_values("entity_id") for _, g in baseline.groupby("volume_band", observed=True)]
quota = [0] * len(parts)
for _ in range(min(400, len(baseline))):
    candidates = [i for i, g in enumerate(parts) if quota[i] < len(g)]
    chosen = min(candidates, key=lambda i: (quota[i], i))
    quota[chosen] += 1
sample = (
    pd.concat(
        [g.sample(n=q, random_state=42) for g, q in zip(parts, quota, strict=True) if q],
        ignore_index=True,
    )
    .sort_values("entity_id")
    .reset_index(drop=True)
)
display(
    pd.DataFrame(
        {
            "tramo": [str(g.volume_band.iloc[0]) for g in parts],
            "poblacion": [len(g) for g in parts],
            "muestra": quota,
        }
    )
)
print("IDs muestreados y variables originales:")
display(
    sample[["entity_id", "volume_band", "award_count", "fraction_0", "fraction_1", "fraction_2"]]
)
cols = meta["scaling"]["columns"]
x_sample = (sample[cols].to_numpy() - np.array(meta["scaling"]["mean"])) / np.array(
    meta["scaling"]["scale"]
)
x_sample[:, 1:] *= meta["scaling"]["weighted_category_factor"]
ward_tree = linkage(x_sample, method="ward", metric="euclidean")
plt.figure(figsize=(12, 5))
dendrogram(ward_tree, truncate_mode="lastp", p=25, show_leaf_counts=True)
plt.title("Ward: últimas 25 fusiones, muestra estratificada")
plt.xlabel("Grupos agregados de la muestra")
plt.ylabel("Altura de fusión Ward")
plt.show()
if meta["kmeans_selected"]:
    sample["ward_label"] = fcluster(
        ward_tree, t=int(meta["kmeans_selected"]["k"]), criterion="maxclust"
    )
    comparison = sample.merge(k_selected[["entity_id", "label"]], on="entity_id")
    print("Muestra:", len(sample), "| grupos Ward:", sample.ward_label.nunique())
    print(
        "ARI Ward / KMeans sobre IDs comunes:",
        adjusted_rand_score(comparison.ward_label, comparison.label),
    )
    display(pd.crosstab(comparison.ward_label, comparison.label))

## 6. Estabilidad de HDBSCAN por remuestreo

Cinco subconjuntos del 80 % se comparan con la ejecución completa solo en los ID compartidos. Reportamos ARI incluyendo el ruido y ARI restringido a los puntos asignados en ambas ejecuciones, junto con el tamaño de esa intersección. La escala permanece ajustada a toda la población: la estabilidad es condicional al preprocesamiento. HDBSCAN puede variar entre plataformas y los vectores idénticos hacen sensible la construcción de densidades.

In [ ]:
import hdbscan
from threadpoolctl import threadpool_limits

if meta["hdbscan_selected"]:
    hconfig = meta["hdbscan_selected"]
    ordered = features.sort_values("entity_id").reset_index(drop=True)
    x_all = (ordered[cols].to_numpy() - np.array(meta["scaling"]["mean"])) / np.array(
        meta["scaling"]["scale"]
    )
    x_all[:, 1:] *= meta["scaling"]["weighted_category_factor"]
    reference = (
        ordered[["entity_id"]]
        .merge(h_selected[["entity_id", "label"]], on="entity_id", validate="one_to_one")
        .label.to_numpy()
    )
    stability_h = []
    with threadpool_limits(limits=1):
        for seed in (1, 7, 21, 42, 100):
            rng = np.random.default_rng(seed)
            idx = np.sort(rng.choice(len(ordered), size=int(0.8 * len(ordered)), replace=False))
            labels = hdbscan.HDBSCAN(
                min_cluster_size=int(hconfig["min_cluster_size"]),
                min_samples=int(hconfig["min_samples"]),
                core_dist_n_jobs=1,
            ).fit_predict(x_all[idx])
            ref = reference[idx]
            both = (ref != -1) & (labels != -1)
            stability_h.append(
                {
                    "seed": seed,
                    "n": len(idx),
                    "noise_fraction": float((labels == -1).mean()),
                    "ari_including_noise": adjusted_rand_score(ref, labels),
                    "both_assigned": int(both.sum()),
                    "ari_both_assigned": adjusted_rand_score(ref[both], labels[both])
                    if both.sum() > 1
                    else None,
                }
            )
    display(pd.DataFrame(stability_h))

## 7. Lectura y límites

- La composición de categorías distingue perfiles dentro de volúmenes similares, pero las tres proporciones son dependientes y no representan gasto.
- HDBSCAN describe densidad en estas cuatro variables; los vectores repetidos y los cambios de escala, parámetros o muestra afectan sus etiquetas. La silueta sobre asignados no es directamente comparable con la silueta de toda la población.
- Ward solo describe la muestra estratificada; el corte se hizo para una comparación exploratoria con KMeans.
- Las señales de distancia y ruido no implican fraude, desempeño ni responsabilidad. La cobertura de la fuente y la identidad no resuelta limitan las conclusiones; consultar el reporte de la Parte II para las cifras e interpretación verificadas.

**Referencia de la corrida Windows:** Ward y KMeans alcanzaron ARI 0,483 en 400 ID. El grupo KMeans de mayor volumen concentró 672 de sus 968 entidades en el tramo superior; los otros tres separan perfiles con medianas de 5–6 adjudicaciones según su composición. HDBSCAN obtuvo ARI de 0,420–0,669 en remuestreos con ruido incluido. De 29 candidatos por distancia, 14 coincidieron con ruido. Las cifras dependen de los datos y del entorno; el reporte conserva el contexto y las diferencias observadas con Docker.